# HW7. NYC Taxi Time-Series Features and a JSONB Event Log in PostgreSQL

Two halves of the same problem. The first builds machine-learning features over a daily
time series with window functions: lag, rolling and expanding, every one of them partitioned
by pickup zone. The second stores a semi-structured event log in `JSONB`, queries it three
different ways, and measures with `EXPLAIN (ANALYZE, BUFFERS)` what a GIN index and an
expression B-tree index actually do to the plans.

All helper tables use the `t7_hw_` prefix and every block starts with `DROP ... IF EXISTS`,
so the notebook is reproducible under **Restart & Run All**.

## 1. Setup and data

### 1.1 Environment

The first cell installs the dependencies and brings up an embedded PostgreSQL through
`pgserver`, so the notebook needs no database of its own. On Colab the current runtime is
Python 3.13, for which `pgserver` has no wheel yet; in that case the 3.12 wheel is retagged
as `abi3` and installed, which works because the package ships a prebuilt server rather than
compiled Python extensions.

In [1]:
import glob, os, subprocess, sys, tempfile

os.environ.setdefault("XDG_RUNTIME_DIR", tempfile.mkdtemp())


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


pip_install("jupysql", "psycopg2-binary", "sqlalchemy", "pandas", "pyarrow")

try:
    import pgserver
except ImportError:
    if sys.version_info >= (3, 13):
        tmp = tempfile.mkdtemp()
        subprocess.check_call([
            sys.executable, "-m", "pip", "download", "pgserver==0.1.4",
            "--no-deps", "--only-binary=:all:", "--python-version", "3.12",
            "-d", tmp, "-q",
        ])
        wheel = glob.glob(os.path.join(tmp, "pgserver-*.whl"))[0]
        fixed = wheel.replace("-cp312-cp312-", "-cp312-abi3-")
        os.rename(wheel, fixed)
        pip_install(fixed)
    else:
        pip_install("pgserver==0.1.4")
    import pgserver

print(f"Python {sys.version.split()[0]} | pgserver ready")

Python 3.11.0 | pgserver ready


In [2]:
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import DBAPIError

PGDATA = Path(tempfile.gettempdir()) / "hw7_pg"
pg = pgserver.get_server(PGDATA, cleanup_mode="stop")
DB_URI = pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1)

probe = create_engine(DB_URI, future=True)
with probe.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())


# True if the server accepts the statement on a fresh connection
def works(statement: str) -> bool:
    try:
        with probe.connect() as conn:
            conn.execute(text(statement))
        return True
    except DBAPIError:
        return False


# The Windows build of pgserver ships without the IANA time zone database, so
# neither named zones nor plain 'UTC' resolve there. Fall back to the equivalent
# POSIX rules, which are built in, and the notebook runs on Colab and locally alike.
SESSION_TZ = "UTC" if works("SET TIME ZONE 'UTC'") else "UTC0"
NYC_TZ = (
    "America/New_York"
    if works("SELECT TIMESTAMP '2024-03-10 12:00' AT TIME ZONE 'America/New_York'")
    else "EST5EDT,M3.2.0,M11.1.0"
)
probe.dispose()

print("session time zone:", SESSION_TZ)
print("pickup time zone :", NYC_TZ)

PostgreSQL 16.2 on x86_64-pc-mingw64, compiled by gcc.exe (x86_64-posix-seh-rev2, Built by MinGW-W64 project) 12.2.0, 64-bit
session time zone: UTC0
pickup time zone : EST5EDT,M3.2.0,M11.1.0


In [3]:
engine = create_engine(DB_URI, future=True, connect_args={"options": f"-c timezone={SESSION_TZ}"})


def q(query: str) -> pd.DataFrame:
    """Run a query and return it as a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)


def explain(query: str) -> None:
    """Print an EXPLAIN (ANALYZE, BUFFERS) plan with its indentation preserved."""
    with engine.connect() as conn:
        plan = conn.execute(text(f"EXPLAIN (ANALYZE, BUFFERS) {query}")).scalars().all()
    print("\n".join(plan))


%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 25
%sql engine

In [4]:
%%sql
SELECT current_setting('server_version') AS pg_version,
       current_setting('TimeZone')       AS session_tz;

pg_version,session_tz
16.2,UTC0


### 1.2 Getting the data — Variant A, the official TLC Parquet

The source is the NYC Taxi & Limousine Commission trip record archive, which is public and
needs no credentials. Three deliberate changes to the template, each of them necessary:

**Two months instead of one.** The template reads `yellow_tripdata_2024-03.parquet` and takes
`head(500_000)`. The Parquet file is ordered by pickup time, so the first 500 000 rows cover
only seven days — 29 February to 6 March. With seven days of history `LAG(trip_count, 30)`
is `NULL` on every row, and the combined feature view, which filters
`WHERE trips_lag_30d IS NOT NULL`, comes out **empty**. February and March together give
60 days, the same horizon the stub fallback in the task is built around.

**A random sample instead of the first rows.** `head()` is a time filter in disguise;
`sample()` with a fixed seed keeps the whole 60-day span and stays reproducible.

**An explicit date window.** The March file contains stray pickup timestamps from 2002 and
from April 2024 — real TLC data quality. Without a window those rows become their own
`ts_day` groups and distort the series.

In [5]:
import shutil
from urllib.request import urlretrieve

MONTHS = ["2024-02", "2024-03"]
SAMPLE_ROWS = 500_000
RANDOM_STATE = 42
PERIOD_START, PERIOD_END = "2024-02-01", "2024-04-01"

COLUMNS = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "passenger_count",
]

frames = []
for month in MONTHS:
    local = Path(tempfile.gettempdir()) / f"yellow_tripdata_{month}.parquet"
    if not local.exists():
        partial, _ = urlretrieve(
            f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{month}.parquet"
        )
        shutil.move(partial, local)
    frames.append(pd.read_parquet(local, columns=COLUMNS))
    print(f"{month}: {len(frames[-1]):>9,} rows  ({local.stat().st_size / 1e6:.0f} MB)")

trips = pd.concat(frames, ignore_index=True)
print(f"\ncombined: {len(trips):,} rows")

# the same cleaning rules the task applies before loading
pickup = trips["tpep_pickup_datetime"]
trips = trips[
    trips["fare_amount"].between(0, 500, inclusive="neither")
    & trips["trip_distance"].between(0, 100, inclusive="neither")
    & trips["PULocationID"].notna()
    & pickup.ge(PERIOD_START)
    & pickup.lt(PERIOD_END)
]
print(f"after cleaning and the date window: {len(trips):,} rows")

trips = trips.sample(n=SAMPLE_ROWS, random_state=RANDOM_STATE)
covered = trips["tpep_pickup_datetime"].dt.date
print(f"sample: {len(trips):,} rows, {covered.nunique()} days "
      f"({covered.min()} to {covered.max()})")

2024-02: 3,007,526 rows  (50 MB)
2024-03: 3,582,628 rows  (60 MB)

combined: 6,590,154 rows


after cleaning and the date window: 6,341,305 rows


sample: 500,000 rows, 60 days (2024-02-01 to 2024-03-31)


In [6]:
trips.to_sql("nyc_taxi_raw", engine, if_exists="replace", index=False, chunksize=10_000)
q("SELECT COUNT(*) AS n_rows FROM nyc_taxi_raw")

,n_rows
0,500000


### 1.3 Daily aggregation — `t7_hw_taxi_daily`

One row per pickup zone per day. This is the table every time-series query below runs on.

In [7]:
%%sql
-- the feature view of a previous run depends on this table, so it goes first
DROP VIEW IF EXISTS v_t7_hw_taxi_features;
DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE;

CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
    COUNT(*)::INTEGER AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
GROUP BY
    "PULocationID"::INTEGER,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day);

ANALYZE t7_hw_taxi_daily;

++
||
++
++

In [8]:
%%sql
SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10;

pickup_zone_id,n_days
1,11
2,1
3,16
4,59
6,3
7,59
8,1
9,6
10,52
11,5


In [9]:
%%sql
SELECT COUNT(*)                                   AS zone_day_rows,
       COUNT(DISTINCT pickup_zone_id)             AS zones,
       COUNT(DISTINCT ts_day)                     AS days,
       MIN(ts_day)                                AS first_day,
       MAX(ts_day)                                AS last_day
FROM t7_hw_taxi_daily;

zone_day_rows,zones,days,first_day,last_day
8623,246,60,2024-02-01,2024-03-31


Well over the 100 rows the task asks for. Coverage is uneven, though: the busy zones report
trips every day, while quiet ones have gaps. That matters for the features, so it is worth
measuring before computing them.

In [10]:
%%sql
SELECT CASE
           WHEN n_days = 60 THEN 'all 60 days'
           WHEN n_days >= 31 THEN '31 to 59 days'
           WHEN n_days >= 8  THEN '8 to 30 days'
           ELSE '7 days or fewer'
       END AS coverage,
       COUNT(*) AS zones
FROM (
    SELECT pickup_zone_id, COUNT(*) AS n_days
    FROM t7_hw_taxi_daily
    GROUP BY pickup_zone_id
) AS per_zone
GROUP BY 1
ORDER BY MIN(n_days) DESC;

coverage,zones
all 60 days,61
31 to 59 days,73
8 to 30 days,81
7 days or fewer,31


### 1.4 A typed table with `TIMESTAMPTZ`

The daily table stores a `DATE`, which is the right grain for the features. For a
production-style record of the instant itself, the pickup is a wall-clock reading in New York
and only becomes a point in time once that zone is attached — which is exactly what
`TIMESTAMPTZ` stores.

In [11]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS t7_hw_taxi_trips_typed"))
    conn.execute(text(f"""
        CREATE TABLE t7_hw_taxi_trips_typed AS
        SELECT
            "PULocationID"::INTEGER AS pickup_zone_id,
            "DOLocationID"::INTEGER AS dropoff_zone_id,
            (tpep_pickup_datetime::TIMESTAMP  AT TIME ZONE '{NYC_TZ}') AS pickup_at,
            (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE '{NYC_TZ}') AS dropoff_at,
            trip_distance::NUMERIC(10, 2) AS trip_distance,
            fare_amount::NUMERIC(10, 2)   AS fare_amount,
            passenger_count::INTEGER      AS passenger_count
        FROM nyc_taxi_raw
        WHERE fare_amount > 0
          AND fare_amount < 500
          AND trip_distance > 0
          AND trip_distance < 100
    """))

q("SELECT pickup_at, pg_typeof(pickup_at) AS pickup_type FROM t7_hw_taxi_trips_typed LIMIT 5")

,pickup_at,pickup_type
0,2024-02-03 19:00:07+00:00,timestamp with time zone
1,2024-03-18 23:49:40+00:00,timestamp with time zone
2,2024-03-01 23:45:42+00:00,timestamp with time zone
3,2024-02-10 21:07:51+00:00,timestamp with time zone
4,2024-03-08 02:18:06+00:00,timestamp with time zone


## 2. Time-series ML features

Four families of features, all of them over `PARTITION BY pickup_zone_id`. Without that
partition a window would run off the end of one zone and into the next — section 2.6 shows
what that costs.

### 2.1 Lag features: 1, 7 and 30 days back

In [12]:
%%sql
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
1,2024-02-04,1,None,None,None
1,2024-02-17,1,1,None,None
1,2024-02-22,1,1,None,None
1,2024-03-08,1,1,None,None
1,2024-03-10,1,1,None,None
1,2024-03-13,1,1,None,None
1,2024-03-16,1,1,None,None
1,2024-03-17,2,1,1,None
1,2024-03-22,1,2,1,None
1,2024-03-24,1,1,1,None


The first rows of every zone carry `NULL`, because there is no history behind them yet.
One caveat worth stating: `LAG(trip_count, 30)` counts **rows**, not calendar days. For a zone
that reports every day the two coincide; for a zone with gaps the 30th row back is further
than 30 days. The rolling windows below avoid this by framing on `RANGE ... INTERVAL`, which
is measured in days rather than rows.

### 2.2 Rolling 7-day average fare

`RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW` is the calendar week including
today. `ts_day` is cast to `TIMESTAMP` so the interval frame is unambiguous.

In [13]:
%%sql
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*)      OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
1,2024-02-04,85.30,85.3000000000000000,1
1,2024-02-17,3.70,3.7000000000000000,1
1,2024-02-22,80.00,41.8500000000000000,2
1,2024-03-08,105.00,105.0000000000000000,1
1,2024-03-10,21.00,63.0000000000000000,2
1,2024-03-13,117.00,81.0000000000000000,3
1,2024-03-16,9.30,49.1000000000000000,3
1,2024-03-17,106.50,77.6000000000000000,3
1,2024-03-22,98.00,71.2666666666666667,3
1,2024-03-24,50.00,74.0000000000000000,2


### 2.3 Rolling 30-day fare volatility

`STDDEV_SAMP` needs at least two rows in the frame, so the first day of each zone is `NULL`
by definition rather than by accident.

In [14]:
%%sql
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
1,2024-02-04,85.30,None,1
1,2024-02-17,3.70,57.6999133448222780,2
1,2024-02-22,80.00,45.6587706068980063,3
1,2024-03-08,105.00,52.7705410243253049,3
1,2024-03-10,21.00,47.9102198561712579,4
1,2024-03-13,117.00,50.5522304156799792,5
1,2024-03-16,9.30,50.6738196705162537,6
1,2024-03-17,106.50,50.0418301214570080,7
1,2024-03-22,98.00,43.6326494180458315,7
1,2024-03-24,50.00,44.7128244094092838,7


### 2.4 Expanding cumulative trips

`ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` accumulates everything the zone has seen
so far, which is the one frame here that never looks forward and so never leaks the future
into a training row.

In [15]:
%%sql
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
1,2024-02-04,1,1,1.00000000000000000000
1,2024-02-17,1,2,1.00000000000000000000
1,2024-02-22,1,3,1.00000000000000000000
1,2024-03-08,1,4,1.00000000000000000000
1,2024-03-10,1,5,1.00000000000000000000
1,2024-03-13,1,6,1.00000000000000000000
1,2024-03-16,1,7,1.00000000000000000000
1,2024-03-17,2,9,1.1250000000000000
1,2024-03-22,1,10,1.1111111111111111
1,2024-03-24,1,11,1.1000000000000000


### 2.5 Combined feature view

All four families in one view. Rows without 30 days of history are dropped, which is what a
training set with complete features looks like.

In [16]:
%%sql
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

        AVG(avg_fare)         OVER w7  AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;

++
||
++
++

In [17]:
%%sql
SELECT *
FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20;

pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
4,2024-03-03,26,17.35,85.38,24,32,5,18.8185714285714286,5.5526853220504298,397,12.8064516129032258
4,2024-03-04,10,27.30,49.95,26,4,15,20.1142857142857143,5.8096186592165618,407,12.7187500000000000
4,2024-03-05,4,24.59,22.43,10,11,26,20.6642857142857143,5.8628857493852656,411,12.4545454545454545
4,2024-03-06,10,21.80,27.98,4,10,19,20.7857142857142857,5.8872531377026046,421,12.3823529411764706
4,2024-03-07,9,17.92,24.02,10,7,3,20.9728571428571429,5.8813697467955683,430,12.2857142857142857
4,2024-03-08,11,19.31,33.90,9,15,5,21.2100000000000000,5.5936762104002793,441,12.2500000000000000
4,2024-03-09,33,17.28,87.71,11,24,3,20.7928571428571429,5.5881289333165406,474,12.8108108108108108
4,2024-03-10,23,17.16,73.19,33,26,3,20.7657142857142857,5.4561547251004945,497,13.0789473684210526
4,2024-03-11,5,17.73,14.15,23,10,12,19.3985714285714286,5.4332770278243446,502,12.8717948717948718
4,2024-03-12,9,23.35,41.00,5,4,23,19.2214285714285714,5.4992161779137735,511,12.7750000000000000


In [18]:
%%sql
SELECT COUNT(*)                       AS training_rows,
       COUNT(DISTINCT pickup_zone_id) AS zones,
       MIN(ts_day)                    AS first_day,
       MAX(ts_day)                    AS last_day
FROM v_t7_hw_taxi_features;

training_rows,zones,first_day,last_day
2968,134,2024-03-02,2024-03-31


### 2.6 What `PARTITION BY` actually prevents

Drop the partition and the window treats all zones as one long series. At the first row of a
zone, `LAG` then returns the last row of the *previous* zone. Nothing fails, nothing warns —
the model simply trains on another zone's history. The query below puts the correct and the
broken value side by side at exactly those boundary rows.

In [19]:
%%sql
WITH lag_compare AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        LAG(trip_count) OVER (PARTITION BY pickup_zone_id ORDER BY ts_day) AS lag_partitioned,
        LAG(trip_count) OVER (ORDER BY pickup_zone_id, ts_day)             AS lag_unpartitioned,
        ROW_NUMBER()    OVER (PARTITION BY pickup_zone_id ORDER BY ts_day) AS rn
    FROM t7_hw_taxi_daily
)
SELECT pickup_zone_id, ts_day, trip_count, lag_partitioned, lag_unpartitioned
FROM lag_compare
WHERE rn = 1
ORDER BY pickup_zone_id
LIMIT 10;

pickup_zone_id,ts_day,trip_count,lag_partitioned,lag_unpartitioned
1,2024-02-04,1,None,None
2,2024-03-20,1,None,1
3,2024-02-09,1,None,1
4,2024-02-01,5,None,1
6,2024-02-09,1,None,25
7,2024-02-01,1,None,1
8,2024-03-22,1,None,11
9,2024-02-12,1,None,1
10,2024-02-01,2,None,1
11,2024-02-08,1,None,2


`lag_partitioned` is `NULL`, as it must be on a zone's first day. `lag_unpartitioned` is a
number — borrowed from a different zone.

## 3. A JSONB event log

A synthetic clickstream with five event types. The point is that the `payload` schema differs
per type: a `page_view` has a URL and a referrer, an `add_to_cart` has a product and an array
of tags, a `recommendation_shown` carries a model name and an array of recommended ids. That
is the situation `JSONB` exists for — one table, several shapes.

In [20]:
%%sql
DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;

++
||
++
++

In [21]:
%%sql
SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;

event_type,n_events
add_to_cart,10000
checkout,10000
click,10000
page_view,10000
recommendation_shown,10000


### 3.1 Containment, `@>`

`@>` asks whether the left document contains the right one. It matches on structure, not on
text, so it is the operator a GIN index can actually serve.

In [22]:
%%sql
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10;

event_id,user_id,event_ts,payload
3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items_count': 9, 'payment_method': 'card'}"
33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items_count': 4, 'payment_method': 'card'}"
48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items_count': 9, 'payment_method': 'card'}"
63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'items_count': 4, 'payment_method': 'card'}"
78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'items_count': 9, 'payment_method': 'card'}"
93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'items_count': 4, 'payment_method': 'card'}"
108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'items_count': 9, 'payment_method': 'card'}"
123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'items_count': 4, 'payment_method': 'card'}"
138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'items_count': 9, 'payment_method': 'card'}"


### 3.2 Key existence, `?` and `?|`

These test **top-level** keys only. A key nested inside an object is invisible to them, which
is also why indexing them needs the default `jsonb_ops` rather than `jsonb_path_ops`.

In [23]:
%%sql
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10;

event_id,event_type,payload
2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quantity': 3, 'product_id': 3}"
7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3, 'product_id': 8}"
12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity': 3, 'product_id': 13}"
17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quantity': 3, 'product_id': 18}"
22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3, 'product_id': 23}"
27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity': 3, 'product_id': 28}"
32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quantity': 3, 'product_id': 33}"
37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3, 'product_id': 38}"
42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity': 3, 'product_id': 43}"
47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quantity': 3, 'product_id': 48}"


In [24]:
%%sql
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10;

event_id,event_type,payload
4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"
9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [10, 11, 12]}"
14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [15, 16, 17]}"
19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [20, 21, 22]}"
24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [25, 26, 27]}"
29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [30, 31, 32]}"
34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [35, 36, 37]}"
39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [40, 41, 42]}"
44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [45, 46, 47]}"
49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [50, 51, 52]}"


### 3.3 JSONPath

`jsonb_path_query` unnests an array into rows, and `jsonb_path_exists` filters by a predicate
evaluated inside the document.

In [25]:
%%sql
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15;

event_id,recommended_pid
4,5
4,6
4,7
9,10
9,11
9,12
14,15
14,16
14,17
19,20


In [26]:
%%sql
SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10;

event_id,order_id,total
253,o_100253,303.00
258,o_100258,308.00
263,o_100263,313.00
268,o_100268,318.00
273,o_100273,323.00
278,o_100278,328.00
283,o_100283,333.00
288,o_100288,338.00
293,o_100293,343.00
298,o_100298,348.00


## 4. GIN index and `EXPLAIN (ANALYZE, BUFFERS)`

Four plans are captured below: containment before and after a GIN index, then path equality
before and after an expression B-tree index.

### 4.1 Containment without any index

In [27]:
%%sql
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;

ANALYZE t7_hw_events;

++
||
++
++

In [28]:
explain("""
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card')
""")

Aggregate  (cost=1762.34..1762.35 rows=1 width=8) (actual time=29.324..29.325 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=4936 width=0) (actual time=0.013..29.175 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=21
Planning Time: 1.883 ms
Execution Time: 29.343 ms


### 4.2 Building the GIN index

`jsonb_path_ops` indexes whole paths rather than every key and value separately. It is smaller
and faster for `@>`, and it is the right choice here because containment is the workload. The
trade is that it cannot serve the `?` key-existence operators at all.

In [29]:
%%sql
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);

ANALYZE t7_hw_events;

SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops')) AS gin_index_size,
       pg_size_pretty(pg_relation_size('t7_hw_events'))                      AS table_size;

gin_index_size,table_size
1144 kB,8000 kB


### 4.3 The same containment query after the index

In [30]:
explain("""
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card')
""")

Aggregate  (cost=1099.44..1099.45 rows=1 width=8) (actual time=3.196..3.196 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=35.99..1090.38 rows=3626 width=0) (actual time=0.443..3.063 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..35.08 rows=3626 width=0) (actual time=0.331..0.331 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=33
Planning Time: 1.765 ms
Execution Time: 3.221 ms


In [31]:
explain("""
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google')
""")

Aggregate  (cost=1080.10..1080.11 rows=1 width=8) (actual time=2.575..2.576 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=31.52..1073.16 rows=2776 width=0) (actual time=0.349..2.472 rows=2500 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..30.83 rows=2776 width=0) (actual time=0.235..0.236 rows=2500 loops=1)
              Index Cond: (payload @> jsonb_build_object('referrer', 'google'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.111 ms
Execution Time: 2.598 ms


### 4.4 Where GIN stops helping: path-specific equality

`payload ->> 'device' = 'mobile'` extracts a value and compares it as text. That is a function
over the column, not a containment test, so the GIN index cannot answer it.

In [32]:
explain("""
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile'
""")

Aggregate  (cost=1750.63..1750.63 rows=1 width=8) (actual time=7.019..7.020 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.016..6.669 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.048 ms
Execution Time: 7.032 ms


### 4.5 The expression index that does help

An index on the expression itself stores exactly the extracted text, so the planner can match
it to the predicate.

In [33]:
%%sql
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));

ANALYZE t7_hw_events;

SELECT pg_size_pretty(pg_relation_size('idx_t7_events_device_btree')) AS btree_index_size;

btree_index_size
360 kB


In [34]:
explain("""
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile'
""")

Aggregate  (cost=1239.79..1239.80 rows=1 width=8) (actual time=1.983..1.984 rows=1 loops=1)
  Buffers: shared hit=1000 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=95.98..1219.25 rows=8218 width=0) (actual time=0.534..1.641 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1000 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..93.92 rows=8218 width=0) (actual time=0.359..0.359 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=22 read=1
Planning Time: 1.628 ms
Execution Time: 2.006 ms


## 5. JSONB against a normalized table

### 5.1 The normalized checkout table

Every `checkout` payload has the same four fields, so it can be flattened into typed columns
with constraints — the schema stops being a convention and becomes something the database
enforces.

In [35]:
%%sql
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id, user_id, event_ts, order_id, total, items_count, payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id',
    (payload ->> 'total')::NUMERIC(10, 2),
    (payload ->> 'items_count')::INTEGER,
    payload ->> 'payment_method'
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;

SELECT COUNT(*) AS n_checkouts FROM t7_hw_checkouts_norm;

n_checkouts
10000


### 5.2 One question, two storage models

Average checkout total per payment method, top three. Identical semantics, different physics.

In [36]:
explain("""
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3
""")

Limit  (cost=1955.30..1955.31 rows=3 width=72) (actual time=8.720..8.721 rows=3 loops=1)
  Buffers: shared hit=1003
  ->  Sort  (cost=1955.30..1969.19 rows=5554 width=72) (actual time=8.719..8.719 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1003
        ->  HashAggregate  (cost=1800.21..1883.52 rows=5554 width=72) (actual time=8.681..8.696 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1650.03 rows=10012 width=132) (actual time=0.010..5.169 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning:
  Buffers: shared hit=34 read=1
Planning Time: 0.162 ms
Execution Time: 8.845 ms


In [37]:
explain("""
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3
""")

Limit  (cost=269.06..269.07 rows=3 width=46) (actual time=2.655..2.656 rows=3 loops=1)
  Buffers: shared hit=94
  ->  Sort  (cost=269.06..269.07 rows=3 width=46) (actual time=2.654..2.654 rows=3 loops=1)
        Sort Key: (avg(total)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=94
        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=2.639..2.641 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=94
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.009..0.737 rows=10000 loops=1)
                    Buffers: shared hit=94
Planning:
  Buffers: shared hit=34
Planning Time: 2.020 ms
Execution Time: 2.680 ms


In [38]:
%%sql
SELECT payment_method, ROUND(AVG(total), 2) AS avg_total, COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;

payment_method,avg_total,n
paypal,300.52,3333
card,300.50,3334
crypto,300.48,3333


### 5.3 Bonus: the same aggregation in MongoDB

Pseudocode for comparison — this is not executed, and it is not PostgreSQL.

```javascript
// MongoDB equivalent: average checkout total per payment_method, top 3
db.events.aggregate([
  {$match: {event_type: "checkout"}},
  {$group: {
    _id: "$payload.payment_method",
    avg_total: {$avg: {$toDouble: "$payload.total"}},
    n: {$sum: 1}
  }},
  {$sort: {avg_total: -1}},
  {$limit: 3}
]);
```

The pipeline reads naturally because the document *is* the storage unit: no extraction step,
no cast from text, no schema to declare. What it gives up is everything around the query —
joins to the curated tables, a transaction spanning several collections, and a `CHECK`
constraint that refuses a fourth payment method. PostgreSQL with `JSONB` keeps the flexible
shape and the relational machinery in one place, which is why a raw clickstream can land in
`JSONB` while the features a model trains on live in typed tables next to it. A document
store or a data lake earns its place when the raw volume is far larger than what a single
PostgreSQL instance should hold.

## 6. Reflection

**Before and after GIN.** The containment query on `payment_method` began as a `Seq Scan` over
50 000 rows: cost 1762.34, 29.343 ms, 3334 rows against an estimate of 4936. After the
`jsonb_path_ops` GIN index it became a `Bitmap Index Scan` feeding a `Bitmap Heap Scan`: cost
1099.44, 3.221 ms, nine times faster. The `referrer` query took the same shape at 2.598 ms.

**Why GIN misses `payload ->> 'device' = 'mobile'`.** GIN indexes the paths and values inside
the document, which is what `@>` tests. `->>` extracts one value as text and the predicate
compares its result, so nothing in the index matches it: the plan stayed a `Seq Scan` at
7.032 ms. The expression index `((payload ->> 'device'))` stores exactly that text and produced
a `Bitmap Index Scan` at 2.006 ms. It also repaired the estimate, which matters more than the
milliseconds: 250 rows expected against 8333 actual became 8218, and a 33-fold miss like that
wrecks join ordering in a larger query.

**The trade.** GIN costs 1144 kB against an 8000 kB table, about 14%, and every insert adds
that row's paths to it, so writes and vacuum pay for the faster `@>` and JSONPath reads.

**JSONB or normalized.** The same aggregation read 132 bytes per row from `t7_hw_events` in
8.845 ms and 11 bytes from `t7_hw_checkouts_norm` in 2.680 ms. Both plans are a `Seq Scan`, so
the gap is row width and the per-row extract-and-cast, not the access method. Raw logs belong
in JSONB, the features a model reads belong in typed columns.

**Adding `refund`.** In JSONB it is one `INSERT` with a new `event_type` and no migration, and
the cost moves to every consumer that must now handle a sixth payload shape. Normalized, it is
a new table and a backfill, but the `CHECK` constraints reject a malformed refund at write time
instead of surfacing it later as a `NULL` feature.

## 7. Final checks

In [39]:
daily = q("SELECT COUNT(*) AS n, COUNT(DISTINCT ts_day) AS days FROM t7_hw_taxi_daily").iloc[0]
assert daily["n"] > 100, "the daily table must hold more than 100 rows"
assert daily["days"] == 60, "two months should give 60 distinct days"

features = q("""
    SELECT COUNT(*) AS n,
           COUNT(trips_lag_1d)  AS lag1,
           COUNT(trips_lag_7d)  AS lag7,
           COUNT(trips_lag_30d) AS lag30,
           COUNT(rolling_avg_fare_7d)  AS roll7,
           COUNT(rolling_std_fare_30d) AS std30,
           COUNT(cumulative_trips)     AS cum
    FROM v_t7_hw_taxi_features
""").iloc[0]
assert features["n"] > 0, "the feature view must not be empty"
assert features["lag30"] == features["n"], "every row of the view must carry a 30-day lag"

events = q("SELECT COUNT(DISTINCT event_type) AS types, COUNT(*) AS n FROM t7_hw_events").iloc[0]
assert events["types"] >= 5, "at least five event types are required"

indexes = set(q("""
    SELECT indexname FROM pg_indexes
    WHERE tablename IN ('t7_hw_events', 't7_hw_checkouts_norm')
""")["indexname"])
assert "idx_t7_events_payload_gin_pathops" in indexes
assert "idx_t7_events_device_btree" in indexes

norm = q("SELECT COUNT(*) AS n FROM t7_hw_checkouts_norm").iloc[0]["n"]
assert norm == q("SELECT COUNT(*) AS n FROM t7_hw_events WHERE event_type = 'checkout'").iloc[0]["n"]

print(f"daily rows      : {daily['n']:,} over {daily['days']} days")
print(f"feature rows    : {features['n']:,}")
print(f"events          : {events['n']:,} across {events['types']} types")
print(f"checkouts       : {norm:,} in both the JSONB log and the normalized table")
print("All checks passed")

daily rows      : 8,623 over 60 days
feature rows    : 2,968
events          : 50,000 across 5 types
checkouts       : 10,000 in both the JSONB log and the normalized table
All checks passed
